In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

In [3]:
N = 35000  # number of customer records we'll generate

customer_id = [f"CUST{100000 + i}" for i in range(N)]

In [4]:
print(len(customer_id))
print(customer_id[:5])

35000
['CUST100000', 'CUST100001', 'CUST100002', 'CUST100003', 'CUST100004']


In [5]:
age = np.random.randint(18, 66, N)

countries = ["USA", "India", "UK", "Canada", "Germany", "Australia",
             "France", "Brazil", "Singapore", "UAE"]
country_weights = [0.28, 0.18, 0.10, 0.08, 0.08, 0.06, 0.06, 0.06, 0.05, 0.05]
country = np.random.choice(countries, N, p=country_weights)

industries = ["Technology", "E-commerce", "Healthcare", "Education",
              "Finance", "Marketing/Media", "Manufacturing", "Retail"]
industry = np.random.choice(industries, N)

company_size = np.random.choice(
    ["Small", "Medium", "Enterprise"], N, p=[0.55, 0.32, 0.13]
)

In [6]:
print(age[:5])
print(pd.Series(country).value_counts())
print(pd.Series(company_size).value_counts())

[56 46 32 60 25]
USA          9824
India        6315
UK           3465
Canada       2863
Germany      2852
France       2092
Brazil       2076
Australia    2067
Singapore    1733
UAE          1713
Name: count, dtype: int64
Small         19237
Medium        11281
Enterprise     4482
Name: count, dtype: int64


In [7]:
plan_probs = {
    "Small":      [0.55, 0.30, 0.12, 0.03],
    "Medium":     [0.20, 0.40, 0.30, 0.10],
    "Enterprise": [0.03, 0.12, 0.35, 0.50],
}
plans = ["Basic", "Standard", "Premium", "Enterprise"]

subscription_plan = np.array([
    np.random.choice(plans, p=plan_probs[cs]) for cs in company_size
])

In [8]:
print(pd.crosstab(company_size, subscription_plan))

col_0       Basic  Enterprise  Premium  Standard
row_0                                           
Enterprise    140        2248     1547       547
Medium       2253        1104     3386      4538
Small       10586         600     2342      5709


In [9]:
start_date = datetime(2022, 1, 1)
end_date = datetime(2025, 12, 31)
date_range_days = (end_date - start_date).days

subscription_date = [
    start_date + timedelta(days=int(np.random.randint(0, date_range_days)))
    for _ in range(N)
]

In [10]:
print(subscription_date[:5])
print(min(subscription_date), max(subscription_date))

[datetime.datetime(2024, 2, 10, 0, 0), datetime.datetime(2024, 5, 4, 0, 0), datetime.datetime(2023, 8, 12, 0, 0), datetime.datetime(2025, 3, 23, 0, 0), datetime.datetime(2025, 3, 23, 0, 0)]
2022-01-01 00:00:00 2025-12-30 00:00:00


In [11]:
base_fee_map = {"Basic": 15, "Standard": 45, "Premium": 99, "Enterprise": 249}

monthly_fee = np.array([
    base_fee_map[p] * np.random.uniform(0.9, 1.15) for p in subscription_plan
]).round(2)

In [12]:


print(pd.DataFrame({"plan": subscription_plan, "fee": monthly_fee}).groupby("plan")["fee"].describe())

              count        mean        std     min      25%      50%  \
plan                                                                   
Basic       12979.0   15.384355   1.079802   13.50   14.460   15.380   
Enterprise   3952.0  255.274598  17.988642  224.11  239.570  255.505   
Premium      7275.0  101.481680   7.156965   89.10   95.225  101.540   
Standard    10794.0   46.088353   3.261885   40.50   43.250   46.080   

                 75%     max  
plan                          
Basic        16.3200   17.25  
Enterprise  270.8525  286.33  
Premium     107.6000  113.85  
Standard     48.8900   51.75  


In [13]:
discount = np.round(np.random.beta(2, 8, N) * 30, 1)

payment_methods = ["Credit Card", "Debit Card", "PayPal", "Bank Transfer", "UPI"]
payment_method = np.random.choice(payment_methods, N, p=[0.45, 0.15, 0.20, 0.10, 0.10])

channels = ["Organic Search", "Paid Ads", "Social Media", "Referral", "Email Campaign", "Affiliate"]
channel_weights = [0.22, 0.24, 0.20, 0.14, 0.12, 0.08]
marketing_channel = np.random.choice(channels, N, p=channel_weights)

In [14]:
print(pd.Series(discount).describe())
print(pd.Series(payment_method).value_counts(normalize=True).round(2))

count    35000.000000
mean         5.996846
std          3.631741
min          0.000000
25%          3.200000
50%          5.400000
75%          8.100000
max         23.800000
dtype: float64
Credit Card      0.45
PayPal           0.20
Debit Card       0.15
Bank Transfer    0.10
UPI              0.10
Name: proportion, dtype: float64


In [15]:
plan_rank = {"Basic": 0, "Standard": 1, "Premium": 2, "Enterprise": 3}
plan_rank_arr = np.array([plan_rank[p] for p in subscription_plan])

login_frequency = np.clip(
    np.random.normal(loc=3 + plan_rank_arr * 1.2, scale=2.0, size=N), 0, 30
).round(1)

usage_hours = np.clip(
    np.random.normal(loc=8 + plan_rank_arr * 6, scale=5, size=N), 0, 200
).round(1)

feature_usage_pct = np.clip(
    np.random.normal(loc=35 + plan_rank_arr * 12, scale=15, size=N), 0, 100
).round(1)

support_tickets = np.random.poisson(lam=np.clip(3 - plan_rank_arr * 0.4, 0.5, None), size=N)

In [16]:
print(pd.DataFrame({"plan": subscription_plan, "usage": usage_hours, "logins": login_frequency, "tickets": support_tickets}).groupby("plan").mean(numeric_only=True))

                usage    logins   tickets
plan                                     
Basic        8.000948  3.070344  2.993605
Enterprise  26.052100  6.598988  1.786437
Premium     20.015079  5.387134  2.223093
Standard    14.014675  4.244580  2.614415


In [17]:
upgrade_count = np.random.poisson(lam=0.3 + plan_rank_arr * 0.1, size=N)
downgrade_count = np.random.poisson(lam=0.15, size=N)

In [18]:
print(pd.DataFrame({"plan": subscription_plan, "upgrades": upgrade_count}).groupby("plan").mean())

            upgrades
plan                
Basic       0.299099
Enterprise  0.577935
Premium     0.499381
Standard    0.393645


In [19]:
tenure_days = np.array([(datetime(2026, 1, 1) - d).days for d in subscription_date])
tenure_months = tenure_days / 30.44

z = (
    -0.05 * usage_hours
    -0.07 * login_frequency
    + 0.18 * support_tickets
    - 0.6 * plan_rank_arr
    - 0.01 * tenure_months
    + 0.15
)
churn_prob = 1 / (1 + np.exp(-z))
churn_prob = np.clip(churn_prob, 0.02, 0.80)
is_churned = np.random.rand(N) < churn_prob

customer_status = np.where(is_churned, "Churned", "Active")

In [20]:
print("Overall churn rate:", round(is_churned.mean()*100, 2), "%")
print(pd.DataFrame({"plan": subscription_plan, "churned": is_churned}).groupby("plan")["churned"].mean())

Overall churn rate: 27.04 %
plan
Basic         0.464828
Enterprise    0.042763
Premium       0.101168
Standard      0.234112
Name: churned, dtype: float64


In [21]:
cancellation_date = []
for i in range(N):
    if is_churned[i]:
        max_extra = max(int(tenure_days[i] * 0.9), 30)
        cdate = subscription_date[i] + timedelta(days=int(np.random.randint(30, max_extra + 30)))
        cdate = min(cdate, datetime(2026, 1, 15))
        cancellation_date.append(cdate)
    else:
        cancellation_date.append(pd.NaT)

renewal_date = [
    d + timedelta(days=365) if not is_churned[i] else pd.NaT
    for i, d in enumerate(subscription_date)
]

In [22]:
print(pd.DataFrame({"status": customer_status, "cancel": cancellation_date}).head(10))
print("Missing cancellation dates (should equal Active count):", pd.isna(cancellation_date).sum())
print("Active count:", (customer_status=="Active").sum())

    status     cancel
0  Churned 2024-11-28
1   Active        NaT
2   Active        NaT
3  Churned 2025-09-10
4   Active        NaT
5   Active        NaT
6   Active        NaT
7   Active        NaT
8   Active        NaT
9   Active        NaT
Missing cancellation dates (should equal Active count): 25535
Active count: 25535


In [23]:
satisfaction = (
    7.5
    - 0.35 * support_tickets
    + 0.015 * usage_hours
    + 0.05 * login_frequency
    - 1.2 * is_churned.astype(int)
    + np.random.normal(0, 0.8, N)
)
customer_satisfaction = np.clip(satisfaction, 1, 10).round(1)

In [24]:
print(pd.DataFrame({"status": customer_status, "satisfaction": customer_satisfaction}).groupby("status").mean(numeric_only=True))

         satisfaction
status               
Active       7.144323
Churned      5.527660


In [25]:
effective_fee = monthly_fee * (1 - discount / 100)

active_months = np.where(
    is_churned,
    np.clip(
        np.array([(c - s).days if not pd.isna(c) else 0
                  for c, s in zip(cancellation_date, subscription_date)]) / 30.44,
        1, None
    ),
    tenure_months
)

lifetime_value = (effective_fee * active_months).round(2)

In [26]:
print(pd.DataFrame({"plan": subscription_plan, "ltv": lifetime_value}).groupby("plan")["ltv"].mean().round(2))

plan
Basic          269.09
Enterprise    5576.27
Premium       2173.40
Standard       928.78
Name: ltv, dtype: float64


In [27]:
df = pd.DataFrame({
    "Customer_ID": customer_id,
    "Age": age,
    "Country": country,
    "Industry": industry,
    "Company_Size": company_size,
    "Subscription_Plan": subscription_plan,
    "Subscription_Date": subscription_date,
    "Renewal_Date": renewal_date,
    "Monthly_Fee": monthly_fee,
    "Discount_Percent": discount,
    "Payment_Method": payment_method,
    "Marketing_Channel": marketing_channel,
    "Support_Tickets": support_tickets,
    "Login_Frequency_Per_Week": login_frequency,
    "Usage_Hours_Monthly": usage_hours,
    "Feature_Usage_Percent": feature_usage_pct,
    "Upgrade_Count": upgrade_count,
    "Downgrade_Count": downgrade_count,
    "Cancellation_Date": cancellation_date,
    "Customer_Status": customer_status,
    "Customer_Satisfaction": customer_satisfaction,
    "Lifetime_Value": lifetime_value,
})

In [28]:
print(df.shape)
print(df.head())
df.info()

(35000, 22)
  Customer_ID  Age    Country         Industry Company_Size Subscription_Plan  \
0  CUST100000   56     Canada  Marketing/Media        Small             Basic   
1  CUST100001   46      India       Healthcare        Small           Premium   
2  CUST100002   32      India          Finance        Small             Basic   
3  CUST100003   60    Germany  Marketing/Media       Medium           Premium   
4  CUST100004   25  Australia       Technology       Medium          Standard   

  Subscription_Date Renewal_Date  Monthly_Fee  Discount_Percent  ...  \
0        2024-02-10          NaT        15.87               7.2  ...   
1        2024-05-04   2025-05-04       110.60              17.4  ...   
2        2023-08-12   2024-08-11        15.27              12.4  ...   
3        2025-03-23          NaT        98.04               4.6  ...   
4        2025-03-23   2026-03-23        40.68               4.2  ...   

  Support_Tickets Login_Frequency_Per_Week  Usage_Hours_Monthly  \
0

In [29]:
# a) missing values in a few columns
for col, frac in [("Customer_Satisfaction", 0.02), ("Industry", 0.01), ("Payment_Method", 0.015)]:
    idx = np.random.choice(df.index, int(len(df) * frac), replace=False)
    df.loc[idx, col] = np.nan

# b) a handful of exact duplicate rows
dup_idx = np.random.choice(df.index, 40, replace=False)
df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

# c) a few negative/impossible outliers to catch in cleaning
out_idx = np.random.choice(df.index, 15, replace=False)
df.loc[out_idx, "Usage_Hours_Monthly"] = -5

In [30]:
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNegative usage hours:", (df["Usage_Hours_Monthly"] < 0).sum())
print("\nNew shape:", df.shape)

Missing values:
 Industry                   350
Renewal_Date              9475
Payment_Method             525
Cancellation_Date        25565
Customer_Satisfaction      702
dtype: int64

Duplicate rows: 40

Negative usage hours: 15

New shape: (35040, 22)


In [31]:
df.to_csv("../data/raw/saas_customers_raw.csv", index=False)
print("Saved:", df.shape)

Saved: (35040, 22)
